# EXP021 FOCUS3D-first V1 candidate audit

Standalone diagnostic: frozen Pilkwang centers and original FOCUS3D instances are converted into one observation layer, then continuation and division candidates are enumerated before any graph solver. No submission is generated.


In [ ]:
from __future__ import annotations

import contextlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
from collections import defaultdict
from pathlib import Path

import numpy as np


def first_existing(paths):
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError("missing path:\n" + "\n".join(map(str, paths)))


COMP_ROOT = first_existing([
    Path('/kaggle/input/competitions/biohub-cell-tracking-during-development'),
    Path('/kaggle/input/biohub-cell-tracking-during-development'),
])
SUPPORT_ROOT = first_existing([
    Path('/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1'),
    Path('/kaggle/input/pilkwang/biohub-tracking-support-pack-50ep-v1'),
])
FOCUS_ROOT = first_existing([
    Path('/kaggle/input/datasets/qiweiyin/focus3d-nuclei-runtime'),
    Path('/kaggle/input/focus3d-nuclei-runtime'),
])

SAMPLE_ID = '6bba_337b1b3a'
MAX_EVAL_FRAMES = 8
MATCH_RADIUS_UM = 7.0
SCALE_UM = np.asarray((1.625, 0.40625, 0.40625), dtype=np.float32)
SAMPLE_PATH = COMP_ROOT / 'train' / f'{SAMPLE_ID}.zarr'
GT_PATH = COMP_ROOT / 'train' / f'{SAMPLE_ID}.geff'
assert SAMPLE_PATH.exists() and GT_PATH.exists()

try:
    import zarr
except Exception:
    wheels = SUPPORT_ROOT / 'wheels'
    packages = [
        'tracksdata', 'zarr==3.2.1', 'numcodecs==0.15.1', 'donfig==0.8.1.post1',
        'geff==1.2.0.1.1', 'geff-spec==1.1.1', 'polars==1.42.0',
        'polars-runtime-32==1.42.0', 'bidict==0.23.1', 'imagecodecs==2026.6.26',
        'rustworkx==0.18.0', 'ilpy==0.6.0', 'pyscipopt==6.2.1',
    ]
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-index', '--no-deps',
         '--find-links', str(wheels), *packages], capture_output=True, text=True,
    )
    if result.returncode:
        raise RuntimeError((result.stdout or '')[-2000:] + (result.stderr or '')[-2000:])
    import zarr

sys.path.insert(0, str(SUPPORT_ROOT / 'repo' / 'src'))
sys.path.insert(0, str(SUPPORT_ROOT / 'repo' / 'scripts'))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('EXP021 requires CUDA')

gt_group = zarr.open_group(str(GT_PATH), mode='r')
gt_times = np.asarray(gt_group['nodes/props/t/values'][:], dtype=np.int64)
EVAL_FRAMES = np.unique(gt_times)[:MAX_EVAL_FRAMES]
PREDICT_MAX_FRAMES = int(EVAL_FRAMES[-1]) + 1
zarr_arr = zarr.open_group(str(SAMPLE_PATH), mode='r')['0']
print('CUDA:', torch.cuda.get_device_name(0))
print('sample:', SAMPLE_ID, 'frames:', EVAL_FRAMES.tolist())

import base64, sys
from pathlib import Path
focus_pkg = Path('/kaggle/working/focus_first')
focus_pkg.mkdir(parents=True, exist_ok=True)
(focus_pkg / '__init__.py').write_bytes(base64.b64decode('IiIiRk9DVVMzRC1maXJzdCBvYnNlcnZhdGlvbiBhbmQgY2FuZGlkYXRlIHByaW1pdGl2ZXMuCgpUaGlzIHBhY2thZ2UgaXMgaW50ZW50aW9uYWxseSBpbmRlcGVuZGVudCBvZiB0aGUgS2FnZ2xlIG5vdGVib29rIHJ1bnRpbWUuICBJdApjYW4gYmUgdGVzdGVkIG9uIHN5bnRoZXRpYyBhcnJheXMgYW5kIGxhdGVyIGNhbGxlZCBmcm9tIHRoZSBHUFUgbm90ZWJvb2suCiIiIgoKZnJvbSAub2JzZXJ2YXRpb25zIGltcG9ydCAoCiAgICBDZW50ZXJPYnNlcnZhdGlvbiwKICAgIEZvY3VzSW5zdGFuY2UsCiAgICBVbmlmaWVkTm9kZSwKICAgIGJ1aWxkX3VuaWZpZWRfbm9kZXMsCiAgICBleHRyYWN0X2ZvY3VzX2luc3RhbmNlcywKICAgIG1hdGNoX2NlbnRlcnNfdG9faW5zdGFuY2VzLAopCmZyb20gLmNhbmRpZGF0ZXMgaW1wb3J0ICgKICAgIENvbnRpbnVhdGlvbkNhbmRpZGF0ZSwKICAgIERpdmlzaW9uQ2FuZGlkYXRlLAogICAgZW51bWVyYXRlX2NvbnRpbnVhdGlvbnMsCiAgICBlbnVtZXJhdGVfZGl2aXNpb25zLAopCmZyb20gLnJ1bnRpbWUgaW1wb3J0IEZvY3VzRnJhbWVPYnNlcnZhdGlvbiwgRm9jdXNPYnNlcnZhdGlvblByb3ZpZGVyCmZyb20gLnBpcGVsaW5lIGltcG9ydCBGcmFtZUNhbmRpZGF0ZUxheWVyLCBidWlsZF9mcmFtZV9jYW5kaWRhdGVfbGF5ZXJzCgpfX2FsbF9fID0gWwogICAgIkNlbnRlck9ic2VydmF0aW9uIiwKICAgICJGb2N1c0luc3RhbmNlIiwKICAgICJVbmlmaWVkTm9kZSIsCiAgICAiYnVpbGRfdW5pZmllZF9ub2RlcyIsCiAgICAiZXh0cmFjdF9mb2N1c19pbnN0YW5jZXMiLAogICAgIm1hdGNoX2NlbnRlcnNfdG9faW5zdGFuY2VzIiwKICAgICJDb250aW51YXRpb25DYW5kaWRhdGUiLAogICAgIkRpdmlzaW9uQ2FuZGlkYXRlIiwKICAgICJlbnVtZXJhdGVfY29udGludWF0aW9ucyIsCiAgICAiZW51bWVyYXRlX2RpdmlzaW9ucyIsCiAgICAiRm9jdXNGcmFtZU9ic2VydmF0aW9uIiwKICAgICJGb2N1c09ic2VydmF0aW9uUHJvdmlkZXIiLAogICAgIkZyYW1lQ2FuZGlkYXRlTGF5ZXIiLAogICAgImJ1aWxkX2ZyYW1lX2NhbmRpZGF0ZV9sYXllcnMiLApdCg=='))
(focus_pkg / 'observations.py').write_bytes(base64.b64decode('IiIiRmlyc3QtZ2VuZXJhdGlvbiB1bmlmaWVkIG9ic2VydmF0aW9uIGxheWVyIGZvciB0aGUgRk9DVVMzRC1maXJzdCBkZXNpZ24uCgpUaGUgbW9kdWxlIGRlbGliZXJhdGVseSBkb2VzIG5vdCB3cml0ZSBhIGdyYXBoIG9yIG1ha2UgYSBzdWJtaXNzaW9uLiAgSXQgdHVybnMKaW5kZXBlbmRlbnQgY2VudGVyIGFuZCBpbnN0YW5jZSBvYnNlcnZhdGlvbnMgaW50byBhdWRpdGFibGUgbm9kZSBwcm9wb3NhbHMuCkZPQ1VTLW9ubHkgcHJvcG9zYWxzIHN0YXkgcHJvdmlzaW9uYWwgdW50aWwgYSBsYXRlciB0ZW1wb3JhbC9nbG9iYWwgc3RhZ2UuCiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzLCBhc2RpY3QKZnJvbSB0eXBpbmcgaW1wb3J0IEl0ZXJhYmxlCgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBzY2lweS5vcHRpbWl6ZSBpbXBvcnQgbGluZWFyX3N1bV9hc3NpZ25tZW50CgoKREVGQVVMVF9TQ0FMRV9VTSA9ICgxLjYyNSwgMC40MDYyNSwgMC40MDYyNSkKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBDZW50ZXJPYnNlcnZhdGlvbjoKICAgICIiIkEgY2VudGVyIGRldGVjdG9yIG9ic2VydmF0aW9uIGluIHZveGVsIGNvb3JkaW5hdGVzLiIiIgoKICAgIG5vZGVfaWQ6IGludAogICAgdDogaW50CiAgICB6OiBmbG9hdAogICAgeTogZmxvYXQKICAgIHg6IGZsb2F0CiAgICBzY29yZTogZmxvYXQgPSAwLjAKICAgIHNvdXJjZTogc3RyID0gInBpbGt3YW5nIgoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHBvaW50KHNlbGYpIC0+IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdOgogICAgICAgIHJldHVybiAoZmxvYXQoc2VsZi56KSwgZmxvYXQoc2VsZi55KSwgZmxvYXQoc2VsZi54KSkKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBGb2N1c0luc3RhbmNlOgogICAgIiIiQ29tcGFjdCBwZXItaW5zdGFuY2UgRk9DVVMzRCBldmlkZW5jZS4KCiAgICBgYG1hc2tgYCBpcyBhIGNyb3BwZWQgYm9vbGVhbiBhcnJheSBpbiBsb2NhbCBiYm94IGNvb3JkaW5hdGVzLiAgS2VlcGluZyBhCiAgICBjcm9wIHJhdGhlciB0aGFuIGEgZnVsbCB2b2x1bWUgbWFrZXMgbGF0ZXIgc3RyZWFtaW5nL2NhY2hpbmcgcHJhY3RpY2FsLgogICAgYGBiYm94YGAgdXNlcyBoYWxmLW9wZW4gdm94ZWwgYm91bmRzIGBgKHowLCB5MCwgeDAsIHoxLCB5MSwgeDEpYGAuCiAgICAiIiIKCiAgICBsYWJlbDogaW50CiAgICB0OiBpbnQKICAgIGNlbnRyb2lkOiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XQogICAgdm9sdW1lOiBpbnQKICAgIGJib3g6IHR1cGxlW2ludCwgaW50LCBpbnQsIGludCwgaW50LCBpbnRdCiAgICBtYXNrOiBucC5uZGFycmF5IHwgTm9uZSA9IE5vbmUKICAgIG1lYW5fY29uZmlkZW5jZTogZmxvYXQgPSAxLjAKICAgIHExMF9jb25maWRlbmNlOiBmbG9hdCA9IDEuMAogICAgYm91bmRhcnlfZnJhY3Rpb246IGZsb2F0ID0gMC4wCgogICAgQHByb3BlcnR5CiAgICBkZWYgcG9pbnQoc2VsZikgLT4gdHVwbGVbZmxvYXQsIGZsb2F0LCBmbG9hdF06CiAgICAgICAgcmV0dXJuIHNlbGYuY2VudHJvaWQKCiAgICBAcHJvcGVydHkKICAgIGRlZiBxdWFsaXR5KHNlbGYpIC0+IGZsb2F0OgogICAgICAgICIiIkEgcmVsYXRpdmUgcXVhbGl0eSBwcmlvciwgbm90IGEgY2FsaWJyYXRlZCBwcm9iYWJpbGl0eS4iIiIKCiAgICAgICAgdm9sdW1lX3Rlcm0gPSBtaW4oMS4wLCBmbG9hdChzZWxmLnZvbHVtZSkgLyA2NC4wKQogICAgICAgIGJvdW5kYXJ5X3Rlcm0gPSBtYXgoMC4wLCAxLjAgLSBmbG9hdChzZWxmLmJvdW5kYXJ5X2ZyYWN0aW9uKSkKICAgICAgICByZXR1cm4gZmxvYXQoc2VsZi5tZWFuX2NvbmZpZGVuY2UgKiAoMC41ICsgMC41ICogdm9sdW1lX3Rlcm0pICogYm91bmRhcnlfdGVybSkKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBVbmlmaWVkTm9kZToKICAgICIiIkEgbm9kZSBwcm9wb3NhbCBhc3NlbWJsZWQgZnJvbSBvbmUgb3IgYm90aCBvYnNlcnZhdGlvbiBzb3VyY2VzLiIiIgoKICAgIHByb3Bvc2FsX2lkOiBzdHIKICAgIHQ6IGludAogICAgejogZmxvYXQKICAgIHk6IGZsb2F0CiAgICB4OiBmbG9hdAogICAga2luZDogc3RyICAjIGNvbnNlbnN1cywgcGlsa3dhbmdfb25seSwgZm9jdXNfb25seQogICAgY2VudGVyX2lkczogdHVwbGVbaW50LCAuLi5dID0gKCkKICAgICMgYGBmb2N1c19sYWJlbGBgIGlzIGRpYWdub3N0aWMgb25seTsgdXNlIGBgZm9jdXNfa2V5YGAgZm9yIGxvb2t1cHMgYmVjYXVzZQogICAgIyBsYWJlbHMgYXJlIHVuaXF1ZSBvbmx5IHdpdGhpbiBhIHNpbmdsZSBmcmFtZS4KICAgIGZvY3VzX2xhYmVsOiBpbnQgfCBOb25lID0gTm9uZQogICAgZm9jdXNfa2V5OiB0dXBsZVtpbnQsIGludF0gfCBOb25lID0gTm9uZQogICAgY2VudGVyX3Njb3JlOiBmbG9hdCA9IDAuMAogICAgZm9jdXNfcXVhbGl0eTogZmxvYXQgPSAwLjAKICAgIG1hdGNoX2Rpc3RhbmNlX3VtOiBmbG9hdCB8IE5vbmUgPSBOb25lCiAgICBwcm92aXNpb25hbDogYm9vbCA9IFRydWUKCiAgICBAcHJvcGVydHkKICAgIGRlZiBwb2ludChzZWxmKSAtPiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XToKICAgICAgICByZXR1cm4gKGZsb2F0KHNlbGYueiksIGZsb2F0KHNlbGYueSksIGZsb2F0KHNlbGYueCkpCgoKZGVmIF9kaXN0YW5jZV91bSgKICAgIGE6IEl0ZXJhYmxlW2Zsb2F0XSwKICAgIGI6IEl0ZXJhYmxlW2Zsb2F0XSwKICAgIHNjYWxlX3VtOiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XSwKKSAtPiBmbG9hdDoKICAgIGRlbHRhID0gbnAuYXNhcnJheSh0dXBsZShhKSwgZHR5cGU9bnAuZmxvYXQ2NCkgLSBucC5hc2FycmF5KHR1cGxlKGIpLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgc2NhbGUgPSBucC5hc2FycmF5KHNjYWxlX3VtLCBkdHlwZT1ucC5mbG9hdDY0KQogICAgcmV0dXJuIGZsb2F0KG5wLnNxcnQobnAuc3VtKChkZWx0YSAqIHNjYWxlKSAqKiAyKSkpCgoKZGVmIF92YWxpZGF0ZV9jb25maWRlbmNlKGNvbmZpZGVuY2VfbWFwOiBucC5uZGFycmF5IHwgTm9uZSwgc2hhcGU6IHR1cGxlW2ludCwgLi4uXSkgLT4gbnAubmRhcnJheSB8IE5vbmU6CiAgICBpZiBjb25maWRlbmNlX21hcCBpcyBOb25lOgogICAgICAgIHJldHVybiBOb25lCiAgICBjb25maWRlbmNlID0gbnAuYXNhcnJheShjb25maWRlbmNlX21hcCwgZHR5cGU9bnAuZmxvYXQzMikKICAgIGlmIGNvbmZpZGVuY2Uuc2hhcGUgIT0gc2hhcGU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImNvbmZpZGVuY2VfbWFwIHNoYXBlIHtjb25maWRlbmNlLnNoYXBlfSAhPSBpbnN0YW5jZV9tYXAgc2hhcGUge3NoYXBlfSIpCiAgICByZXR1cm4gY29uZmlkZW5jZQoKCmRlZiBleHRyYWN0X2ZvY3VzX2luc3RhbmNlcygKICAgIGluc3RhbmNlX21hcDogbnAubmRhcnJheSwKICAgICosCiAgICB0OiBpbnQsCiAgICBjb25maWRlbmNlX21hcDogbnAubmRhcnJheSB8IE5vbmUgPSBOb25lLAogICAgbWluX3ZvbHVtZTogaW50ID0gMSwKKSAtPiBsaXN0W0ZvY3VzSW5zdGFuY2VdOgogICAgIiIiRXh0cmFjdCBjb21wYWN0IGluc3RhbmNlcyBhbmQgcHJlc2VydmUgRk9DVVMgY29uZmlkZW5jZSBldmlkZW5jZS4iIiIKCiAgICBhcnIgPSBucC5hc2FycmF5KGluc3RhbmNlX21hcCkKICAgIGlmIGFyci5uZGltICE9IDM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImluc3RhbmNlX21hcCBtdXN0IGJlIDMtRCwgZ290IHthcnIuc2hhcGV9IikKICAgIGNvbmZpZGVuY2UgPSBfdmFsaWRhdGVfY29uZmlkZW5jZShjb25maWRlbmNlX21hcCwgYXJyLnNoYXBlKQogICAgcmVzdWx0OiBsaXN0W0ZvY3VzSW5zdGFuY2VdID0gW10KICAgIHNoYXBlID0gbnAuYXNhcnJheShhcnIuc2hhcGUsIGR0eXBlPW5wLmludDY0KQogICAgZm9yIHJhd19sYWJlbCBpbiBucC51bmlxdWUoYXJyKToKICAgICAgICBsYWJlbCA9IGludChyYXdfbGFiZWwpCiAgICAgICAgaWYgbGFiZWwgPD0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBjb29yZHMgPSBucC5hcmd3aGVyZShhcnIgPT0gcmF3X2xhYmVsKQogICAgICAgIHZvbHVtZSA9IGludChjb29yZHMuc2hhcGVbMF0pCiAgICAgICAgaWYgdm9sdW1lIDwgaW50KG1pbl92b2x1bWUpOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG1pbnMgPSBjb29yZHMubWluKGF4aXM9MCkuYXN0eXBlKGludCkKICAgICAgICBtYXhzID0gKGNvb3Jkcy5tYXgoYXhpcz0wKSArIDEpLmFzdHlwZShpbnQpCiAgICAgICAgbG9jYWwgPSBjb29yZHMgLSBtaW5zCiAgICAgICAgY3JvcF9zaGFwZSA9IHR1cGxlKChtYXhzIC0gbWlucykudG9saXN0KCkpCiAgICAgICAgbWFzayA9IG5wLnplcm9zKGNyb3Bfc2hhcGUsIGR0eXBlPWJvb2wpCiAgICAgICAgbWFza1t0dXBsZShsb2NhbC5UKV0gPSBUcnVlCiAgICAgICAgY2VudHJvaWQgPSB0dXBsZShmbG9hdCh2KSBmb3IgdiBpbiBjb29yZHMubWVhbihheGlzPTApKQogICAgICAgIGlmIGNvbmZpZGVuY2UgaXMgTm9uZToKICAgICAgICAgICAgbWVhbl9jb25maWRlbmNlID0gcTEwX2NvbmZpZGVuY2UgPSAxLjAKICAgICAgICBlbHNlOgogICAgICAgICAgICB2YWx1ZXMgPSBjb25maWRlbmNlW2FyciA9PSByYXdfbGFiZWxdLmFzdHlwZShucC5mbG9hdDY0KQogICAgICAgICAgICBtZWFuX2NvbmZpZGVuY2UgPSBmbG9hdChucC5tZWFuKHZhbHVlcykpCiAgICAgICAgICAgIHExMF9jb25maWRlbmNlID0gZmxvYXQobnAucXVhbnRpbGUodmFsdWVzLCAwLjEwKSkKICAgICAgICB0b3VjaGVzID0gbnAuYW55KChjb29yZHMgPT0gMCkgfCAoY29vcmRzID09IChzaGFwZSAtIDEpKSwgYXhpcz0xKQogICAgICAgIGJvdW5kYXJ5X2ZyYWN0aW9uID0gZmxvYXQobnAubWVhbih0b3VjaGVzKSkgaWYgdm9sdW1lIGVsc2UgMC4wCiAgICAgICAgcmVzdWx0LmFwcGVuZCgKICAgICAgICAgICAgRm9jdXNJbnN0YW5jZSgKICAgICAgICAgICAgICAgIGxhYmVsPWxhYmVsLAogICAgICAgICAgICAgICAgdD1pbnQodCksCiAgICAgICAgICAgICAgICBjZW50cm9pZD1jZW50cm9pZCwKICAgICAgICAgICAgICAgIHZvbHVtZT12b2x1bWUsCiAgICAgICAgICAgICAgICBiYm94PXR1cGxlKGludCh2KSBmb3IgdiBpbiAoKm1pbnMsICptYXhzKSksCiAgICAgICAgICAgICAgICBtYXNrPW1hc2ssCiAgICAgICAgICAgICAgICBtZWFuX2NvbmZpZGVuY2U9bWVhbl9jb25maWRlbmNlLAogICAgICAgICAgICAgICAgcTEwX2NvbmZpZGVuY2U9cTEwX2NvbmZpZGVuY2UsCiAgICAgICAgICAgICAgICBib3VuZGFyeV9mcmFjdGlvbj1ib3VuZGFyeV9mcmFjdGlvbiwKICAgICAgICAgICAgKQogICAgICAgICkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgbWF0Y2hfY2VudGVyc190b19pbnN0YW5jZXMoCiAgICBjZW50ZXJzOiBJdGVyYWJsZVtDZW50ZXJPYnNlcnZhdGlvbl0sCiAgICBpbnN0YW5jZXM6IEl0ZXJhYmxlW0ZvY3VzSW5zdGFuY2VdLAogICAgKiwKICAgIHNjYWxlX3VtOiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XSA9IERFRkFVTFRfU0NBTEVfVU0sCiAgICByYWRpdXNfdW06IGZsb2F0ID0gNy4wLAopIC0+IGRpY3RbaW50LCB0dXBsZVtpbnQsIGZsb2F0XV06CiAgICAiIiJSZXR1cm4gb25lLXRvLW9uZSBjZW50ZXItaWQgLT4gKGluc3RhbmNlLWxhYmVsLCBkaXN0YW5jZSkgbWF0Y2hlcy4iIiIKCiAgICBjZW50ZXJzID0gbGlzdChjZW50ZXJzKQogICAgaW5zdGFuY2VzID0gbGlzdChpbnN0YW5jZXMpCiAgICBpZiBub3QgY2VudGVycyBvciBub3QgaW5zdGFuY2VzOgogICAgICAgIHJldHVybiB7fQogICAgY29zdHMgPSBucC5hc2FycmF5KAogICAgICAgIFtbX2Rpc3RhbmNlX3VtKGNlbnRlci5wb2ludCwgaW5zdGFuY2UucG9pbnQsIHNjYWxlX3VtKSBmb3IgaW5zdGFuY2UgaW4gaW5zdGFuY2VzXSBmb3IgY2VudGVyIGluIGNlbnRlcnNdLAogICAgICAgIGR0eXBlPW5wLmZsb2F0NjQsCiAgICApCiAgICByb3dzLCBjb2xzID0gbGluZWFyX3N1bV9hc3NpZ25tZW50KGNvc3RzKQogICAgbWF0Y2hlczogZGljdFtpbnQsIHR1cGxlW2ludCwgZmxvYXRdXSA9IHt9CiAgICBmb3Igcm93LCBjb2wgaW4gemlwKHJvd3MsIGNvbHMpOgogICAgICAgIGRpc3RhbmNlID0gZmxvYXQoY29zdHNbaW50KHJvdyksIGludChjb2wpXSkKICAgICAgICBpZiBkaXN0YW5jZSA8PSBmbG9hdChyYWRpdXNfdW0pOgogICAgICAgICAgICBtYXRjaGVzW2ludChjZW50ZXJzW2ludChyb3cpXS5ub2RlX2lkKV0gPSAoaW50KGluc3RhbmNlc1tpbnQoY29sKV0ubGFiZWwpLCBkaXN0YW5jZSkKICAgIHJldHVybiBtYXRjaGVzCgoKZGVmIGJ1aWxkX3VuaWZpZWRfbm9kZXMoCiAgICBjZW50ZXJzOiBJdGVyYWJsZVtDZW50ZXJPYnNlcnZhdGlvbl0sCiAgICBpbnN0YW5jZXM6IEl0ZXJhYmxlW0ZvY3VzSW5zdGFuY2VdLAogICAgKiwKICAgIHNjYWxlX3VtOiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XSA9IERFRkFVTFRfU0NBTEVfVU0sCiAgICBtYXRjaF9yYWRpdXNfdW06IGZsb2F0ID0gNy4wLAogICAgYWNjZXB0X2ZvY3VzX3F1YWxpdHk6IGZsb2F0ID0gMC4wLAopIC0+IGxpc3RbVW5pZmllZE5vZGVdOgogICAgIiIiQnVpbGQgY29uc2Vuc3VzIGFuZCBwcm92aXNpb25hbCBzb3VyY2Utc3BlY2lmaWMgbm9kZSBwcm9wb3NhbHMuCgogICAgVGhpcyBmdW5jdGlvbiBkb2VzIG5vdCBzaWxlbnRseSBkaXNjYXJkIEZPQ1VTLW9ubHkgaW5zdGFuY2VzLiAgVGhleSBhcmUKICAgIGV4cGxpY2l0bHkgbWFya2VkIHByb3Zpc2lvbmFsIHNvIGEgbGF0ZXIgdGVtcG9yYWwvZ2xvYmFsIHN0YWdlIGNhbiBkZWNpZGUKICAgIHdoZXRoZXIgdGhlaXIgbm9kZS1jb3VudCBwZW5hbHR5IGlzIGp1c3RpZmllZC4KICAgICIiIgoKICAgIGNlbnRlcnMgPSBsaXN0KGNlbnRlcnMpCiAgICBpbnN0YW5jZXMgPSBsaXN0KGluc3RhbmNlcykKICAgIGJ5X2xhYmVsID0ge2luc3RhbmNlLmxhYmVsOiBpbnN0YW5jZSBmb3IgaW5zdGFuY2UgaW4gaW5zdGFuY2VzfQogICAgbWF0Y2hlcyA9IG1hdGNoX2NlbnRlcnNfdG9faW5zdGFuY2VzKAogICAgICAgIGNlbnRlcnMsIGluc3RhbmNlcywgc2NhbGVfdW09c2NhbGVfdW0sIHJhZGl1c191bT1tYXRjaF9yYWRpdXNfdW0KICAgICkKICAgIG1hdGNoZWRfbGFiZWxzID0ge2xhYmVsIGZvciBsYWJlbCwgXyBpbiBtYXRjaGVzLnZhbHVlcygpfQogICAgcmVzdWx0OiBsaXN0W1VuaWZpZWROb2RlXSA9IFtdCiAgICBmb3IgY2VudGVyIGluIGNlbnRlcnM6CiAgICAgICAgbWF0Y2hlZCA9IG1hdGNoZXMuZ2V0KGludChjZW50ZXIubm9kZV9pZCkpCiAgICAgICAgaWYgbWF0Y2hlZCBpcyBOb25lOgogICAgICAgICAgICByZXN1bHQuYXBwZW5kKAogICAgICAgICAgICAgICAgVW5pZmllZE5vZGUoCiAgICAgICAgICAgICAgICAgICAgcHJvcG9zYWxfaWQ9ZiJwOntjZW50ZXIudH06e2NlbnRlci5ub2RlX2lkfSIsCiAgICAgICAgICAgICAgICAgICAgdD1pbnQoY2VudGVyLnQpLAogICAgICAgICAgICAgICAgICAgIHo9ZmxvYXQoY2VudGVyLnopLAogICAgICAgICAgICAgICAgICAgIHk9ZmxvYXQoY2VudGVyLnkpLAogICAgICAgICAgICAgICAgICAgIHg9ZmxvYXQoY2VudGVyLngpLAogICAgICAgICAgICAgICAgICAgIGtpbmQ9InBpbGt3YW5nX29ubHkiLAogICAgICAgICAgICAgICAgICAgIGNlbnRlcl9pZHM9KGludChjZW50ZXIubm9kZV9pZCksKSwKICAgICAgICAgICAgICAgICAgICBjZW50ZXJfc2NvcmU9ZmxvYXQoY2VudGVyLnNjb3JlKSwKICAgICAgICAgICAgICAgICAgICBwcm92aXNpb25hbD1GYWxzZSwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGxhYmVsLCBkaXN0YW5jZSA9IG1hdGNoZWQKICAgICAgICBpbnN0YW5jZSA9IGJ5X2xhYmVsW2xhYmVsXQogICAgICAgIHJlc3VsdC5hcHBlbmQoCiAgICAgICAgICAgIFVuaWZpZWROb2RlKAogICAgICAgICAgICAgICAgcHJvcG9zYWxfaWQ9ZiJjOntjZW50ZXIudH06e2NlbnRlci5ub2RlX2lkfTp7bGFiZWx9IiwKICAgICAgICAgICAgICAgIHQ9aW50KGNlbnRlci50KSwKICAgICAgICAgICAgICAgIHo9ZmxvYXQoY2VudGVyLnopLAogICAgICAgICAgICAgICAgeT1mbG9hdChjZW50ZXIueSksCiAgICAgICAgICAgICAgICB4PWZsb2F0KGNlbnRlci54KSwKICAgICAgICAgICAgICAgIGtpbmQ9ImNvbnNlbnN1cyIsCiAgICAgICAgICAgICAgICBjZW50ZXJfaWRzPShpbnQoY2VudGVyLm5vZGVfaWQpLCksCiAgICAgICAgICAgICAgICBmb2N1c19sYWJlbD1pbnQobGFiZWwpLAogICAgICAgICAgICAgICAgZm9jdXNfa2V5PShpbnQoaW5zdGFuY2UudCksIGludChsYWJlbCkpLAogICAgICAgICAgICAgICAgY2VudGVyX3Njb3JlPWZsb2F0KGNlbnRlci5zY29yZSksCiAgICAgICAgICAgICAgICBmb2N1c19xdWFsaXR5PWZsb2F0KGluc3RhbmNlLnF1YWxpdHkpLAogICAgICAgICAgICAgICAgbWF0Y2hfZGlzdGFuY2VfdW09ZmxvYXQoZGlzdGFuY2UpLAogICAgICAgICAgICAgICAgcHJvdmlzaW9uYWw9RmFsc2UsCiAgICAgICAgICAgICkKICAgICAgICApCiAgICBmb3IgaW5zdGFuY2UgaW4gaW5zdGFuY2VzOgogICAgICAgIGlmIGluc3RhbmNlLmxhYmVsIGluIG1hdGNoZWRfbGFiZWxzIG9yIGluc3RhbmNlLnF1YWxpdHkgPCBmbG9hdChhY2NlcHRfZm9jdXNfcXVhbGl0eSk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgcmVzdWx0LmFwcGVuZCgKICAgICAgICAgICAgVW5pZmllZE5vZGUoCiAgICAgICAgICAgICAgICBwcm9wb3NhbF9pZD1mImY6e2luc3RhbmNlLnR9OntpbnN0YW5jZS5sYWJlbH0iLAogICAgICAgICAgICAgICAgdD1pbnQoaW5zdGFuY2UudCksCiAgICAgICAgICAgICAgICB6PWZsb2F0KGluc3RhbmNlLmNlbnRyb2lkWzBdKSwKICAgICAgICAgICAgICAgIHk9ZmxvYXQoaW5zdGFuY2UuY2VudHJvaWRbMV0pLAogICAgICAgICAgICAgICAgeD1mbG9hdChpbnN0YW5jZS5jZW50cm9pZFsyXSksCiAgICAgICAgICAgICAgICBraW5kPSJmb2N1c19vbmx5IiwKICAgICAgICAgICAgICAgIGZvY3VzX2xhYmVsPWludChpbnN0YW5jZS5sYWJlbCksCiAgICAgICAgICAgICAgICBmb2N1c19rZXk9KGludChpbnN0YW5jZS50KSwgaW50KGluc3RhbmNlLmxhYmVsKSksCiAgICAgICAgICAgICAgICBmb2N1c19xdWFsaXR5PWZsb2F0KGluc3RhbmNlLnF1YWxpdHkpLAogICAgICAgICAgICAgICAgcHJvdmlzaW9uYWw9VHJ1ZSwKICAgICAgICAgICAgKQogICAgICAgICkKICAgIHJldHVybiByZXN1bHQKCgpkZWYgbm9kZV90b19kaWN0KG5vZGU6IFVuaWZpZWROb2RlKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgICIiIkpTT04tZnJpZW5kbHkgcmVwcmVzZW50YXRpb24gZm9yIGF1ZGl0IG1hbmlmZXN0cy4iIiIKCiAgICByZXR1cm4gYXNkaWN0KG5vZGUpCg=='))
(focus_pkg / 'candidates.py').write_bytes(base64.b64decode('IiIiSGlnaC1yZWNhbGwgY29udGludWF0aW9uIGFuZCBkaXZpc2lvbiBwcm9wb3NhbCBnZW5lcmF0aW9uLgoKVGhlIGNhbmRpZGF0ZXMgYXJlIGRlbGliZXJhdGVseSBpbmRlcGVuZGVudCBvZiB0aGUgY3VycmVudGx5IHNlbGVjdGVkIGdyYXBoLgpJbiBwYXJ0aWN1bGFyLCBhIGRhdWdodGVyIGlzIGVsaWdpYmxlIGZvciBhIGRpdmlzaW9uIHRyaXBsZSBldmVuIHdoZW4gYW5vdGhlcgpwYXJlbnQgY3VycmVudGx5IG93bnMgaXQgaW4gdGhlIG9sZCBJTFAgb3V0cHV0LgoiIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgYXNkaWN0CmZyb20gaXRlcnRvb2xzIGltcG9ydCBjb21iaW5hdGlvbnMKZnJvbSB0eXBpbmcgaW1wb3J0IENhbGxhYmxlLCBJdGVyYWJsZQoKaW1wb3J0IG51bXB5IGFzIG5wCgpmcm9tIC5vYnNlcnZhdGlvbnMgaW1wb3J0IEZvY3VzSW5zdGFuY2UsIFVuaWZpZWROb2RlLCBERUZBVUxUX1NDQUxFX1VNCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgQ29udGludWF0aW9uQ2FuZGlkYXRlOgogICAgc291cmNlX2lkOiBzdHIKICAgIHRhcmdldF9pZDogc3RyCiAgICBkaXN0YW5jZV91bTogZmxvYXQKICAgIGZvY3VzX292ZXJsYXA6IGZsb2F0IHwgTm9uZQogICAgZm9jdXNfdm9sdW1lX3JhdGlvOiBmbG9hdCB8IE5vbmUKICAgIHNvdXJjZV9raW5kOiBzdHIKICAgIHRhcmdldF9raW5kOiBzdHIKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBEaXZpc2lvbkNhbmRpZGF0ZToKICAgIHBhcmVudF9pZDogc3RyCiAgICBkYXVnaHRlcjFfaWQ6IHN0cgogICAgZGF1Z2h0ZXIyX2lkOiBzdHIKICAgIHBhcmVudF9kaXN0YW5jZV91bTogZmxvYXQKICAgIGRhdWdodGVyX2Rpc3RhbmNlX3VtOiBmbG9hdAogICAgdm9sdW1lX2JhbGFuY2U6IGZsb2F0IHwgTm9uZQogICAgcGFyZW50X3VuaW9uX292ZXJsYXA6IGZsb2F0IHwgTm9uZQogICAgZGF1Z2h0ZXJfc2VwYXJhdGlvbl9zY29yZTogZmxvYXQKCgpkZWYgX2Rpc3RhbmNlX3VtKGE6IFVuaWZpZWROb2RlLCBiOiBVbmlmaWVkTm9kZSwgc2NhbGVfdW06IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdKSAtPiBmbG9hdDoKICAgIGRlbHRhID0gbnAuYXNhcnJheShhLnBvaW50LCBkdHlwZT1ucC5mbG9hdDY0KSAtIG5wLmFzYXJyYXkoYi5wb2ludCwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIHNjYWxlID0gbnAuYXNhcnJheShzY2FsZV91bSwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgIHJldHVybiBmbG9hdChucC5zcXJ0KG5wLnN1bSgoZGVsdGEgKiBzY2FsZSkgKiogMikpKQoKCmRlZiBfYmJveF9pbnRlcnNlY3Rpb24oCiAgICBhOiBGb2N1c0luc3RhbmNlLAogICAgYjogRm9jdXNJbnN0YW5jZSwKICAgIHNoaWZ0X2E6IHR1cGxlW2ludCwgaW50LCBpbnRdID0gKDAsIDAsIDApLAopIC0+IHR1cGxlW3NsaWNlLCBzbGljZSwgc2xpY2UsIHNsaWNlLCBzbGljZSwgc2xpY2VdIHwgTm9uZToKICAgICIiIlJldHVybiBsb2NhbCBzbGljZXMgZm9yIHR3byBtYXNrcyBhZnRlciBzaGlmdGluZyBhIGJ5IGludGVnZXIgdm94ZWxzLiIiIgoKICAgIGF6MCwgYXkwLCBheDAsIGF6MSwgYXkxLCBheDEgPSBhLmJib3gKICAgIGJ6MCwgYnkwLCBieDAsIGJ6MSwgYnkxLCBieDEgPSBiLmJib3gKICAgIHN6LCBzeSwgc3ggPSBzaGlmdF9hCiAgICBhejAsIGF6MSA9IGF6MCArIHN6LCBhejEgKyBzegogICAgYXkwLCBheTEgPSBheTAgKyBzeSwgYXkxICsgc3kKICAgIGF4MCwgYXgxID0gYXgwICsgc3gsIGF4MSArIHN4CiAgICBsbyA9IChtYXgoYXowLCBiejApLCBtYXgoYXkwLCBieTApLCBtYXgoYXgwLCBieDApKQogICAgaGkgPSAobWluKGF6MSwgYnoxKSwgbWluKGF5MSwgYnkxKSwgbWluKGF4MSwgYngxKSkKICAgIGlmIGFueShsZWZ0ID49IHJpZ2h0IGZvciBsZWZ0LCByaWdodCBpbiB6aXAobG8sIGhpKSk6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIGFfc2xpY2VzID0gdHVwbGUoc2xpY2UobGVmdCAtIG9yaWdpbiwgcmlnaHQgLSBvcmlnaW4pIGZvciBsZWZ0LCByaWdodCwgb3JpZ2luIGluIHppcChsbywgaGksIChhejAsIGF5MCwgYXgwKSkpCiAgICBiX3NsaWNlcyA9IHR1cGxlKHNsaWNlKGxlZnQgLSBvcmlnaW4sIHJpZ2h0IC0gb3JpZ2luKSBmb3IgbGVmdCwgcmlnaHQsIG9yaWdpbiBpbiB6aXAobG8sIGhpLCAoYnowLCBieTAsIGJ4MCkpKQogICAgcmV0dXJuICgqYV9zbGljZXMsICpiX3NsaWNlcykKCgpkZWYgbWFza19vdmVybGFwKAogICAgc291cmNlOiBGb2N1c0luc3RhbmNlIHwgTm9uZSwKICAgIHRhcmdldDogRm9jdXNJbnN0YW5jZSB8IE5vbmUsCiAgICBzb3VyY2Vfbm9kZTogVW5pZmllZE5vZGUsCiAgICB0YXJnZXRfbm9kZTogVW5pZmllZE5vZGUsCikgLT4gZmxvYXQgfCBOb25lOgogICAgIiIiQ29tcHV0ZSBhIHRyYW5zbGF0aW9uLWF3YXJlIGNyb3BwZWQtbWFzayBJb1Ugd2l0aG91dCBQeXRob24gdm94ZWwgc2V0cy4iIiIKCiAgICBpZiBzb3VyY2UgaXMgTm9uZSBvciB0YXJnZXQgaXMgTm9uZSBvciBzb3VyY2UubWFzayBpcyBOb25lIG9yIHRhcmdldC5tYXNrIGlzIE5vbmU6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIHNoaWZ0ID0gdHVwbGUoaW50KHJvdW5kKHRhcmdldF9ub2RlLnBvaW50W2ldIC0gc291cmNlX25vZGUucG9pbnRbaV0pKSBmb3IgaSBpbiByYW5nZSgzKSkKICAgIHNsaWNlcyA9IF9iYm94X2ludGVyc2VjdGlvbihzb3VyY2UsIHRhcmdldCwgc2hpZnQpCiAgICBpZiBzbGljZXMgaXMgTm9uZToKICAgICAgICByZXR1cm4gMC4wCiAgICBzcmMgPSBzb3VyY2UubWFza1tzbGljZXNbMF0sIHNsaWNlc1sxXSwgc2xpY2VzWzJdXQogICAgZHN0ID0gdGFyZ2V0Lm1hc2tbc2xpY2VzWzNdLCBzbGljZXNbNF0sIHNsaWNlc1s1XV0KICAgIGludGVyc2VjdGlvbiA9IGludChucC5jb3VudF9ub256ZXJvKHNyYyAmIGRzdCkpCiAgICB1bmlvbiA9IGludChucC5jb3VudF9ub256ZXJvKHNyYyB8IGRzdCkpCiAgICByZXR1cm4gZmxvYXQoaW50ZXJzZWN0aW9uIC8gdW5pb24pIGlmIHVuaW9uIGVsc2UgMC4wCgoKZGVmIF9pbnN0YW5jZV9mb3Jfbm9kZSgKICAgIG5vZGU6IFVuaWZpZWROb2RlLAogICAgaW5zdGFuY2VzX2J5X2xhYmVsOiBkaWN0W29iamVjdCwgRm9jdXNJbnN0YW5jZV0sCikgLT4gRm9jdXNJbnN0YW5jZSB8IE5vbmU6CiAgICAiIiJSZXNvbHZlIGEgZnJhbWUtbG9jYWwgRk9DVVMgbGFiZWwgd2l0aG91dCBjcm9zcy1mcmFtZSBjb2xsaXNpb25zLiIiIgoKICAgIGlmIG5vZGUuZm9jdXNfa2V5IGlzIG5vdCBOb25lIGFuZCBub2RlLmZvY3VzX2tleSBpbiBpbnN0YW5jZXNfYnlfbGFiZWw6CiAgICAgICAgcmV0dXJuIGluc3RhbmNlc19ieV9sYWJlbFtub2RlLmZvY3VzX2tleV0KICAgIHJldHVybiBOb25lCgoKZGVmIGVudW1lcmF0ZV9jb250aW51YXRpb25zKAogICAgc291cmNlX25vZGVzOiBJdGVyYWJsZVtVbmlmaWVkTm9kZV0sCiAgICB0YXJnZXRfbm9kZXM6IEl0ZXJhYmxlW1VuaWZpZWROb2RlXSwKICAgICosCiAgICBpbnN0YW5jZXNfYnlfbGFiZWw6IGRpY3Rbb2JqZWN0LCBGb2N1c0luc3RhbmNlXSB8IE5vbmUgPSBOb25lLAogICAgbWF4X2Rpc3RhbmNlX3VtOiBmbG9hdCA9IDE0LjAsCiAgICBzY2FsZV91bTogdHVwbGVbZmxvYXQsIGZsb2F0LCBmbG9hdF0gPSBERUZBVUxUX1NDQUxFX1VNLAopIC0+IGxpc3RbQ29udGludWF0aW9uQ2FuZGlkYXRlXToKICAgICIiIkVudW1lcmF0ZSBoaWdoLXJlY2FsbCB0IC0+IHQrMSBjYW5kaWRhdGVzIGZyb20gYWxsIG5vZGUgcHJvcG9zYWxzLiIiIgoKICAgIGluc3RhbmNlc19ieV9sYWJlbCA9IGluc3RhbmNlc19ieV9sYWJlbCBvciB7fQogICAgcmVzdWx0OiBsaXN0W0NvbnRpbnVhdGlvbkNhbmRpZGF0ZV0gPSBbXQogICAgZm9yIHNvdXJjZSBpbiBzb3VyY2Vfbm9kZXM6CiAgICAgICAgZm9yIHRhcmdldCBpbiB0YXJnZXRfbm9kZXM6CiAgICAgICAgICAgIGlmIGludCh0YXJnZXQudCkgIT0gaW50KHNvdXJjZS50KSArIDE6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBkaXN0YW5jZSA9IF9kaXN0YW5jZV91bShzb3VyY2UsIHRhcmdldCwgc2NhbGVfdW0pCiAgICAgICAgICAgIGlmIGRpc3RhbmNlID4gZmxvYXQobWF4X2Rpc3RhbmNlX3VtKToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIHNvdXJjZV9pbnN0YW5jZSA9IF9pbnN0YW5jZV9mb3Jfbm9kZShzb3VyY2UsIGluc3RhbmNlc19ieV9sYWJlbCkKICAgICAgICAgICAgdGFyZ2V0X2luc3RhbmNlID0gX2luc3RhbmNlX2Zvcl9ub2RlKHRhcmdldCwgaW5zdGFuY2VzX2J5X2xhYmVsKQogICAgICAgICAgICBvdmVybGFwID0gbWFza19vdmVybGFwKHNvdXJjZV9pbnN0YW5jZSwgdGFyZ2V0X2luc3RhbmNlLCBzb3VyY2UsIHRhcmdldCkKICAgICAgICAgICAgdm9sdW1lX3JhdGlvID0gTm9uZQogICAgICAgICAgICBpZiBzb3VyY2VfaW5zdGFuY2UgaXMgbm90IE5vbmUgYW5kIHRhcmdldF9pbnN0YW5jZSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHZvbHVtZV9yYXRpbyA9IGZsb2F0KAogICAgICAgICAgICAgICAgICAgIG1pbihzb3VyY2VfaW5zdGFuY2Uudm9sdW1lLCB0YXJnZXRfaW5zdGFuY2Uudm9sdW1lKQogICAgICAgICAgICAgICAgICAgIC8gbWF4KHNvdXJjZV9pbnN0YW5jZS52b2x1bWUsIHRhcmdldF9pbnN0YW5jZS52b2x1bWUsIDEpCiAgICAgICAgICAgICAgICApCiAgICAgICAgICAgIHJlc3VsdC5hcHBlbmQoCiAgICAgICAgICAgICAgICBDb250aW51YXRpb25DYW5kaWRhdGUoCiAgICAgICAgICAgICAgICAgICAgc291cmNlX2lkPXNvdXJjZS5wcm9wb3NhbF9pZCwKICAgICAgICAgICAgICAgICAgICB0YXJnZXRfaWQ9dGFyZ2V0LnByb3Bvc2FsX2lkLAogICAgICAgICAgICAgICAgICAgIGRpc3RhbmNlX3VtPWRpc3RhbmNlLAogICAgICAgICAgICAgICAgICAgIGZvY3VzX292ZXJsYXA9b3ZlcmxhcCwKICAgICAgICAgICAgICAgICAgICBmb2N1c192b2x1bWVfcmF0aW89dm9sdW1lX3JhdGlvLAogICAgICAgICAgICAgICAgICAgIHNvdXJjZV9raW5kPXNvdXJjZS5raW5kLAogICAgICAgICAgICAgICAgICAgIHRhcmdldF9raW5kPXRhcmdldC5raW5kLAogICAgICAgICAgICAgICAgKQogICAgICAgICAgICApCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIF91bmlvbl9vdmVybGFwKAogICAgcGFyZW50OiBGb2N1c0luc3RhbmNlLAogICAgZDE6IEZvY3VzSW5zdGFuY2UsCiAgICBkMjogRm9jdXNJbnN0YW5jZSwKICAgIHBhcmVudF9ub2RlOiBVbmlmaWVkTm9kZSwKICAgIGQxX25vZGU6IFVuaWZpZWROb2RlLAogICAgZDJfbm9kZTogVW5pZmllZE5vZGUsCikgLT4gZmxvYXQgfCBOb25lOgogICAgIiIiTWVhc3VyZSBkYXVnaHRlciBzdXBwb3J0IGZvciB0d28gdHJhbnNsYXRlZCBjb3BpZXMgb2YgdGhlIHBhcmVudCBtYXNrLiIiIgoKICAgIGlmIGFueShpbnN0YW5jZS5tYXNrIGlzIE5vbmUgZm9yIGluc3RhbmNlIGluIChwYXJlbnQsIGQxLCBkMikpOgogICAgICAgIHJldHVybiBOb25lCiAgICAjIFRoZSBmaXJzdCBpbXBsZW1lbnRhdGlvbiB1c2VzIGEgY29uc2VydmF0aXZlIGNvbW1vbiBjcm9wLiAgSWYgdGhlCiAgICAjIHRyYW5zbGF0ZWQgcGFyZW50IGxpZXMgb3V0c2lkZSBlaXRoZXIgZGF1Z2h0ZXIgYmJveCwgdGhlIG92ZXJsYXAgaXMgemVyby4KICAgIHNoaWZ0cyA9IFsKICAgICAgICB0dXBsZShpbnQocm91bmQoZDFfbm9kZS5wb2ludFtpXSAtIHBhcmVudF9ub2RlLnBvaW50W2ldKSkgZm9yIGkgaW4gcmFuZ2UoMykpLAogICAgICAgIHR1cGxlKGludChyb3VuZChkMl9ub2RlLnBvaW50W2ldIC0gcGFyZW50X25vZGUucG9pbnRbaV0pKSBmb3IgaSBpbiByYW5nZSgzKSksCiAgICBdCiAgICBjb3ZlcmFnZXMgPSBbXQogICAgZm9yIGRhdWdodGVyLCBkYXVnaHRlcl9ub2RlLCBzaGlmdCBpbiAoKGQxLCBkMV9ub2RlLCBzaGlmdHNbMF0pLCAoZDIsIGQyX25vZGUsIHNoaWZ0c1sxXSkpOgogICAgICAgIHNsaWNlcyA9IF9iYm94X2ludGVyc2VjdGlvbihwYXJlbnQsIGRhdWdodGVyLCBzaGlmdCkKICAgICAgICBpZiBzbGljZXMgaXMgTm9uZToKICAgICAgICAgICAgY292ZXJhZ2VzLmFwcGVuZCgwLjApCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgdHJhbnNsYXRlZF9wYXJlbnQgPSBwYXJlbnQubWFza1tzbGljZXNbMF0sIHNsaWNlc1sxXSwgc2xpY2VzWzJdXQogICAgICAgIGRhdWdodGVyX21hc2sgPSBkYXVnaHRlci5tYXNrW3NsaWNlc1szXSwgc2xpY2VzWzRdLCBzbGljZXNbNV1dCiAgICAgICAgY292ZXJhZ2VzLmFwcGVuZChmbG9hdChucC5jb3VudF9ub256ZXJvKHRyYW5zbGF0ZWRfcGFyZW50ICYgZGF1Z2h0ZXJfbWFzaykpKQogICAgcGFyZW50X3ZvbHVtZSA9IG1heChmbG9hdChwYXJlbnQudm9sdW1lKSwgMS4wKQogICAgIyBUaGUgdW5pb24gaXMgYm91bmRlZCBieSB0aGUgc3VtIG9mIHRoZSB0d28gcGFyZW50IGNvcGllcy4gIFRoaXMgaXMgYQogICAgIyBjYW5kaWRhdGUgZmVhdHVyZSwgbm90IGEgaGFyZCBwaHlzaWNhbCBjb25zZXJ2YXRpb24gbGF3LgogICAgcmV0dXJuIGZsb2F0KG1pbigxLjAsIHN1bShjb3ZlcmFnZXMpIC8gbWF4KHBhcmVudF92b2x1bWUgKiAyLjAsIDEuMCkpKQoKCmRlZiBlbnVtZXJhdGVfZGl2aXNpb25zKAogICAgcGFyZW50X25vZGVzOiBJdGVyYWJsZVtVbmlmaWVkTm9kZV0sCiAgICBkYXVnaHRlcl9ub2RlczogSXRlcmFibGVbVW5pZmllZE5vZGVdLAogICAgKiwKICAgIGluc3RhbmNlc19ieV9sYWJlbDogZGljdFtvYmplY3QsIEZvY3VzSW5zdGFuY2VdIHwgTm9uZSA9IE5vbmUsCiAgICBtYXhfcGFyZW50X2Rpc3RhbmNlX3VtOiBmbG9hdCA9IDE0LjAsCiAgICBtYXhfZGF1Z2h0ZXJfZGlzdGFuY2VfdW06IGZsb2F0ID0gMTQuMCwKICAgIG1pbl9kYXVnaHRlcl9kaXN0YW5jZV91bTogZmxvYXQgPSAwLjAsCiAgICBzY2FsZV91bTogdHVwbGVbZmxvYXQsIGZsb2F0LCBmbG9hdF0gPSBERUZBVUxUX1NDQUxFX1VNLAopIC0+IGxpc3RbRGl2aXNpb25DYW5kaWRhdGVdOgogICAgIiIiRW51bWVyYXRlIGV4cGxpY2l0IHBhcmVudC9kYXVnaHRlciBwYWlycyBiZWZvcmUgYW55IGdyYXBoIGlzIHNlbGVjdGVkLiIiIgoKICAgIGluc3RhbmNlc19ieV9sYWJlbCA9IGluc3RhbmNlc19ieV9sYWJlbCBvciB7fQogICAgcmVzdWx0OiBsaXN0W0RpdmlzaW9uQ2FuZGlkYXRlXSA9IFtdCiAgICBkYXVnaHRlcnMgPSBsaXN0KGRhdWdodGVyX25vZGVzKQogICAgZm9yIHBhcmVudCBpbiBwYXJlbnRfbm9kZXM6CiAgICAgICAgY2FuZGlkYXRlczogbGlzdFtVbmlmaWVkTm9kZV0gPSBbXQogICAgICAgIGZvciBkYXVnaHRlciBpbiBkYXVnaHRlcnM6CiAgICAgICAgICAgIGlmIGludChkYXVnaHRlci50KSAhPSBpbnQocGFyZW50LnQpICsgMToKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlmIF9kaXN0YW5jZV91bShwYXJlbnQsIGRhdWdodGVyLCBzY2FsZV91bSkgPD0gZmxvYXQobWF4X3BhcmVudF9kaXN0YW5jZV91bSk6CiAgICAgICAgICAgICAgICBjYW5kaWRhdGVzLmFwcGVuZChkYXVnaHRlcikKICAgICAgICBmb3IgZDEsIGQyIGluIGNvbWJpbmF0aW9ucyhjYW5kaWRhdGVzLCAyKToKICAgICAgICAgICAgZGF1Z2h0ZXJfZGlzdGFuY2UgPSBfZGlzdGFuY2VfdW0oZDEsIGQyLCBzY2FsZV91bSkKICAgICAgICAgICAgaWYgZGF1Z2h0ZXJfZGlzdGFuY2UgPiBmbG9hdChtYXhfZGF1Z2h0ZXJfZGlzdGFuY2VfdW0pOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaWYgZGF1Z2h0ZXJfZGlzdGFuY2UgPCBmbG9hdChtaW5fZGF1Z2h0ZXJfZGlzdGFuY2VfdW0pOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgaTEgPSBfaW5zdGFuY2VfZm9yX25vZGUoZDEsIGluc3RhbmNlc19ieV9sYWJlbCkKICAgICAgICAgICAgaTIgPSBfaW5zdGFuY2VfZm9yX25vZGUoZDIsIGluc3RhbmNlc19ieV9sYWJlbCkKICAgICAgICAgICAgaXAgPSBfaW5zdGFuY2VfZm9yX25vZGUocGFyZW50LCBpbnN0YW5jZXNfYnlfbGFiZWwpCiAgICAgICAgICAgIGJhbGFuY2UgPSBOb25lCiAgICAgICAgICAgIGlmIGkxIGlzIG5vdCBOb25lIGFuZCBpMiBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGJhbGFuY2UgPSBmbG9hdChtaW4oaTEudm9sdW1lLCBpMi52b2x1bWUpIC8gbWF4KGkxLnZvbHVtZSwgaTIudm9sdW1lLCAxKSkKICAgICAgICAgICAgb3ZlcmxhcCA9IF91bmlvbl9vdmVybGFwKGlwLCBpMSwgaTIsIHBhcmVudCwgZDEsIGQyKSBpZiBpcCBhbmQgaTEgYW5kIGkyIGVsc2UgTm9uZQogICAgICAgICAgICBzZXBhcmF0aW9uID0gZmxvYXQobWluKDEuMCwgZGF1Z2h0ZXJfZGlzdGFuY2UgLyBtYXgoZmxvYXQobWF4X2RhdWdodGVyX2Rpc3RhbmNlX3VtKSwgMWUtNikpKQogICAgICAgICAgICByZXN1bHQuYXBwZW5kKAogICAgICAgICAgICAgICAgRGl2aXNpb25DYW5kaWRhdGUoCiAgICAgICAgICAgICAgICAgICAgcGFyZW50X2lkPXBhcmVudC5wcm9wb3NhbF9pZCwKICAgICAgICAgICAgICAgICAgICBkYXVnaHRlcjFfaWQ9ZDEucHJvcG9zYWxfaWQsCiAgICAgICAgICAgICAgICAgICAgZGF1Z2h0ZXIyX2lkPWQyLnByb3Bvc2FsX2lkLAogICAgICAgICAgICAgICAgICAgIHBhcmVudF9kaXN0YW5jZV91bT1tYXgoX2Rpc3RhbmNlX3VtKHBhcmVudCwgZDEsIHNjYWxlX3VtKSwgX2Rpc3RhbmNlX3VtKHBhcmVudCwgZDIsIHNjYWxlX3VtKSksCiAgICAgICAgICAgICAgICAgICAgZGF1Z2h0ZXJfZGlzdGFuY2VfdW09ZGF1Z2h0ZXJfZGlzdGFuY2UsCiAgICAgICAgICAgICAgICAgICAgdm9sdW1lX2JhbGFuY2U9YmFsYW5jZSwKICAgICAgICAgICAgICAgICAgICBwYXJlbnRfdW5pb25fb3ZlcmxhcD1vdmVybGFwLAogICAgICAgICAgICAgICAgICAgIGRhdWdodGVyX3NlcGFyYXRpb25fc2NvcmU9c2VwYXJhdGlvbiwKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBjYW5kaWRhdGVfdG9fZGljdChjYW5kaWRhdGU6IENvbnRpbnVhdGlvbkNhbmRpZGF0ZSB8IERpdmlzaW9uQ2FuZGlkYXRlKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgIHJldHVybiBhc2RpY3QoY2FuZGlkYXRlKQo='))
(focus_pkg / 'pipeline.py').write_bytes(base64.b64decode('IiIiRnJhbWUtd2lzZSBvcmNoZXN0cmF0aW9uIGZvciB0aGUgRVhQMDIxIFYxIGNhbmRpZGF0ZSBsYXllci4iIiIKCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcwpmcm9tIHR5cGluZyBpbXBvcnQgSXRlcmFibGUKCmZyb20gLmNhbmRpZGF0ZXMgaW1wb3J0ICgKICAgIENvbnRpbnVhdGlvbkNhbmRpZGF0ZSwKICAgIERpdmlzaW9uQ2FuZGlkYXRlLAogICAgZW51bWVyYXRlX2NvbnRpbnVhdGlvbnMsCiAgICBlbnVtZXJhdGVfZGl2aXNpb25zLAopCmZyb20gLm9ic2VydmF0aW9ucyBpbXBvcnQgKAogICAgQ2VudGVyT2JzZXJ2YXRpb24sCiAgICBGb2N1c0luc3RhbmNlLAogICAgVW5pZmllZE5vZGUsCiAgICBidWlsZF91bmlmaWVkX25vZGVzLAogICAgREVGQVVMVF9TQ0FMRV9VTSwKKQoKCkBkYXRhY2xhc3MoZnJvemVuPVRydWUpCmNsYXNzIEZyYW1lQ2FuZGlkYXRlTGF5ZXI6CiAgICAiIiJBbGwgc291cmNlIHByb3Bvc2FscyBhbmQgb3V0Z29pbmcgZXZlbnQgY2FuZGlkYXRlcyBmb3Igb25lIGZyYW1lLiIiIgoKICAgIHQ6IGludAogICAgbm9kZXM6IHR1cGxlW1VuaWZpZWROb2RlLCAuLi5dCiAgICBjb250aW51YXRpb25zX3RvX25leHQ6IHR1cGxlW0NvbnRpbnVhdGlvbkNhbmRpZGF0ZSwgLi4uXSA9ICgpCiAgICBkaXZpc2lvbnNfdG9fbmV4dDogdHVwbGVbRGl2aXNpb25DYW5kaWRhdGUsIC4uLl0gPSAoKQoKCmRlZiBidWlsZF9mcmFtZV9jYW5kaWRhdGVfbGF5ZXJzKAogICAgY2VudGVyc19ieV90OiBkaWN0W2ludCwgSXRlcmFibGVbQ2VudGVyT2JzZXJ2YXRpb25dXSwKICAgIGluc3RhbmNlc19ieV90OiBkaWN0W2ludCwgSXRlcmFibGVbRm9jdXNJbnN0YW5jZV1dLAogICAgKiwKICAgIHNjYWxlX3VtOiB0dXBsZVtmbG9hdCwgZmxvYXQsIGZsb2F0XSA9IERFRkFVTFRfU0NBTEVfVU0sCiAgICBtYXRjaF9yYWRpdXNfdW06IGZsb2F0ID0gNy4wLAogICAgbWF4X2Rpc3RhbmNlX3VtOiBmbG9hdCA9IDE0LjAsCiAgICBtYXhfZGF1Z2h0ZXJfZGlzdGFuY2VfdW06IGZsb2F0ID0gMTQuMCwKICAgIG1pbl9kYXVnaHRlcl9kaXN0YW5jZV91bTogZmxvYXQgPSAwLjAsCiAgICBmb2N1c19vbmx5X3F1YWxpdHlfZmxvb3I6IGZsb2F0ID0gMC4wLAopIC0+IGxpc3RbRnJhbWVDYW5kaWRhdGVMYXllcl06CiAgICAiIiJCdWlsZCBjYW5kaWRhdGVzIGZyb20gZXZlcnkgb2JzZXJ2YXRpb24sIGluZGVwZW5kZW50IG9mIG9sZCBncmFwaCBlZGdlcy4iIiIKCiAgICBmcmFtZV9pZHMgPSBzb3J0ZWQoc2V0KGludCh0KSBmb3IgdCBpbiBjZW50ZXJzX2J5X3QpIHwgc2V0KGludCh0KSBmb3IgdCBpbiBpbnN0YW5jZXNfYnlfdCkpCiAgICBub2Rlc19ieV90OiBkaWN0W2ludCwgbGlzdFtVbmlmaWVkTm9kZV1dID0ge30KICAgIGFsbF9pbnN0YW5jZXM6IGRpY3RbdHVwbGVbaW50LCBpbnRdLCBGb2N1c0luc3RhbmNlXSA9IHt9CiAgICBmb3IgdCBpbiBmcmFtZV9pZHM6CiAgICAgICAgY2VudGVycyA9IGxpc3QoY2VudGVyc19ieV90LmdldCh0LCAoKSkpCiAgICAgICAgaW5zdGFuY2VzID0gbGlzdChpbnN0YW5jZXNfYnlfdC5nZXQodCwgKCkpKQogICAgICAgIGlmIGFueShpbnQoY2VudGVyLnQpICE9IHQgZm9yIGNlbnRlciBpbiBjZW50ZXJzKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImNlbnRlciBvYnNlcnZhdGlvbiBoYXMgd3JvbmcgZnJhbWUgZm9yIHQ9e3R9IikKICAgICAgICBpZiBhbnkoaW50KGluc3RhbmNlLnQpICE9IHQgZm9yIGluc3RhbmNlIGluIGluc3RhbmNlcyk6CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJGT0NVUyBpbnN0YW5jZSBoYXMgd3JvbmcgZnJhbWUgZm9yIHQ9e3R9IikKICAgICAgICBub2Rlc19ieV90W3RdID0gYnVpbGRfdW5pZmllZF9ub2RlcygKICAgICAgICAgICAgY2VudGVycywKICAgICAgICAgICAgaW5zdGFuY2VzLAogICAgICAgICAgICBzY2FsZV91bT1zY2FsZV91bSwKICAgICAgICAgICAgbWF0Y2hfcmFkaXVzX3VtPW1hdGNoX3JhZGl1c191bSwKICAgICAgICAgICAgYWNjZXB0X2ZvY3VzX3F1YWxpdHk9Zm9jdXNfb25seV9xdWFsaXR5X2Zsb29yLAogICAgICAgICkKICAgICAgICBmb3IgaW5zdGFuY2UgaW4gaW5zdGFuY2VzOgogICAgICAgICAgICBhbGxfaW5zdGFuY2VzWyhpbnQoaW5zdGFuY2UudCksIGludChpbnN0YW5jZS5sYWJlbCkpXSA9IGluc3RhbmNlCgogICAgbGF5ZXJzOiBsaXN0W0ZyYW1lQ2FuZGlkYXRlTGF5ZXJdID0gW10KICAgIGZvciBpbmRleCwgdCBpbiBlbnVtZXJhdGUoZnJhbWVfaWRzKToKICAgICAgICBub2RlcyA9IG5vZGVzX2J5X3RbdF0KICAgICAgICBpZiBpbmRleCArIDEgPj0gbGVuKGZyYW1lX2lkcykgb3IgZnJhbWVfaWRzW2luZGV4ICsgMV0gIT0gdCArIDE6CiAgICAgICAgICAgIGxheWVycy5hcHBlbmQoRnJhbWVDYW5kaWRhdGVMYXllcih0PXQsIG5vZGVzPXR1cGxlKG5vZGVzKSkpCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgbmV4dF9ub2RlcyA9IG5vZGVzX2J5X3RbdCArIDFdCiAgICAgICAgY29udGludWF0aW9ucyA9IGVudW1lcmF0ZV9jb250aW51YXRpb25zKAogICAgICAgICAgICBub2RlcywKICAgICAgICAgICAgbmV4dF9ub2RlcywKICAgICAgICAgICAgaW5zdGFuY2VzX2J5X2xhYmVsPWFsbF9pbnN0YW5jZXMsCiAgICAgICAgICAgIG1heF9kaXN0YW5jZV91bT1tYXhfZGlzdGFuY2VfdW0sCiAgICAgICAgICAgIHNjYWxlX3VtPXNjYWxlX3VtLAogICAgICAgICkKICAgICAgICBkaXZpc2lvbnMgPSBlbnVtZXJhdGVfZGl2aXNpb25zKAogICAgICAgICAgICBub2RlcywKICAgICAgICAgICAgbmV4dF9ub2RlcywKICAgICAgICAgICAgaW5zdGFuY2VzX2J5X2xhYmVsPWFsbF9pbnN0YW5jZXMsCiAgICAgICAgICAgIG1heF9wYXJlbnRfZGlzdGFuY2VfdW09bWF4X2Rpc3RhbmNlX3VtLAogICAgICAgICAgICBtYXhfZGF1Z2h0ZXJfZGlzdGFuY2VfdW09bWF4X2RhdWdodGVyX2Rpc3RhbmNlX3VtLAogICAgICAgICAgICBtaW5fZGF1Z2h0ZXJfZGlzdGFuY2VfdW09bWluX2RhdWdodGVyX2Rpc3RhbmNlX3VtLAogICAgICAgICAgICBzY2FsZV91bT1zY2FsZV91bSwKICAgICAgICApCiAgICAgICAgbGF5ZXJzLmFwcGVuZCgKICAgICAgICAgICAgRnJhbWVDYW5kaWRhdGVMYXllcigKICAgICAgICAgICAgICAgIHQ9dCwKICAgICAgICAgICAgICAgIG5vZGVzPXR1cGxlKG5vZGVzKSwKICAgICAgICAgICAgICAgIGNvbnRpbnVhdGlvbnNfdG9fbmV4dD10dXBsZShjb250aW51YXRpb25zKSwKICAgICAgICAgICAgICAgIGRpdmlzaW9uc190b19uZXh0PXR1cGxlKGRpdmlzaW9ucyksCiAgICAgICAgICAgICkKICAgICAgICApCiAgICByZXR1cm4gbGF5ZXJzCgo='))
(focus_pkg / 'audit.py').write_bytes(base64.b64decode('IiIiTWFjaGluZS1yZWFkYWJsZSBzdW1tYXJpZXMgZm9yIHRoZSBFWFAwMjEgY2FuZGlkYXRlLWxheWVyIGF1ZGl0LiIiIgoKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgQ291bnRlcgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBhc2RpY3QKZnJvbSB0eXBpbmcgaW1wb3J0IEl0ZXJhYmxlCgpmcm9tIC5jYW5kaWRhdGVzIGltcG9ydCBDb250aW51YXRpb25DYW5kaWRhdGUsIERpdmlzaW9uQ2FuZGlkYXRlCmZyb20gLm9ic2VydmF0aW9ucyBpbXBvcnQgVW5pZmllZE5vZGUKCgpkZWYgc3VtbWFyaXplX2ZyYW1lKAogICAgbm9kZXM6IEl0ZXJhYmxlW1VuaWZpZWROb2RlXSwKICAgIGNvbnRpbnVhdGlvbnM6IEl0ZXJhYmxlW0NvbnRpbnVhdGlvbkNhbmRpZGF0ZV0sCiAgICBkaXZpc2lvbnM6IEl0ZXJhYmxlW0RpdmlzaW9uQ2FuZGlkYXRlXSwKKSAtPiBkaWN0W3N0ciwgb2JqZWN0XToKICAgICIiIlJldHVybiBjb3VudHMgc3BsaXQgYnkgc291cmNlIGtpbmQsIHdpdGhvdXQgcHJldGVuZGluZyB0aGV5IGFyZSBDVi4iIiIKCiAgICBub2RlcyA9IGxpc3Qobm9kZXMpCiAgICBjb250aW51YXRpb25zID0gbGlzdChjb250aW51YXRpb25zKQogICAgZGl2aXNpb25zID0gbGlzdChkaXZpc2lvbnMpCiAgICByZXR1cm4gewogICAgICAgICJub2RlX2NvdW50IjogbGVuKG5vZGVzKSwKICAgICAgICAibm9kZV9raW5kX2NvdW50cyI6IGRpY3QoQ291bnRlcihub2RlLmtpbmQgZm9yIG5vZGUgaW4gbm9kZXMpKSwKICAgICAgICAicHJvdmlzaW9uYWxfbm9kZV9jb3VudCI6IHN1bShib29sKG5vZGUucHJvdmlzaW9uYWwpIGZvciBub2RlIGluIG5vZGVzKSwKICAgICAgICAiY29udGludWF0aW9uX2NvdW50IjogbGVuKGNvbnRpbnVhdGlvbnMpLAogICAgICAgICJjb250aW51YXRpb25fdGFyZ2V0X2tpbmRfY291bnRzIjogZGljdChDb3VudGVyKGNhbmRpZGF0ZS50YXJnZXRfa2luZCBmb3IgY2FuZGlkYXRlIGluIGNvbnRpbnVhdGlvbnMpKSwKICAgICAgICAiZGl2aXNpb25fY291bnQiOiBsZW4oZGl2aXNpb25zKSwKICAgICAgICAiZGl2aXNpb25fZm9jdXNfZmVhdHVyZV9jb3VudCI6IHN1bSgKICAgICAgICAgICAgY2FuZGlkYXRlLnBhcmVudF91bmlvbl9vdmVybGFwIGlzIG5vdCBOb25lIGZvciBjYW5kaWRhdGUgaW4gZGl2aXNpb25zCiAgICAgICAgKSwKICAgICAgICAiZGl2aXNpb25fY2FuZGlkYXRlcyI6IFthc2RpY3QoY2FuZGlkYXRlKSBmb3IgY2FuZGlkYXRlIGluIGRpdmlzaW9uc10sCiAgICB9Cgo='))
(focus_pkg / 'runtime.py').write_bytes(base64.b64decode('IiIiT25lLXBhc3MgRk9DVVMzRCBydW50aW1lIGFkYXB0ZXIgZm9yIEVYUDAyMS4KClVubGlrZSB0aGUgbGVnYWN5IGBgRm9jdXNNYXNrUHJvdmlkZXJgYCwgdGhpcyBhZGFwdGVyIGtlZXBzIHRoZSBmaW5hbApjb25maWRlbmNlIG1hcCBhbmQgcnVuIG1ldGFkYXRhIHRvZ2V0aGVyIHdpdGggdGhlIGluc3RhbmNlIG1hcC4gIEl0IGlzIGxhenksCmZyYW1lLWNhY2hlZCwgYW5kIGhhcyBubyBncmFwaCBtdXRhdGlvbiBzaWRlIGVmZmVjdHMuICBUaGUgS2FnZ2xlIG5vdGVib29rIGNhbgp3cmFwIGBgZ2V0YGAgd2l0aCBpdHMgb3duIGltYWdlIGxvYWRlci4KIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBvcwppbXBvcnQgc3lzCmZyb20gdHlwaW5nIGltcG9ydCBDYWxsYWJsZSwgQW55CgppbXBvcnQgbnVtcHkgYXMgbnAKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBGb2N1c0ZyYW1lT2JzZXJ2YXRpb246CiAgICBkYXRhc2V0OiBzdHIKICAgIHQ6IGludAogICAgaW5zdGFuY2VfbWFwOiBucC5uZGFycmF5CiAgICBjb25maWRlbmNlX21hcDogbnAubmRhcnJheSB8IE5vbmUKICAgIGxvZ19pbmZvOiBkaWN0W3N0ciwgQW55XQoKCmNsYXNzIEZvY3VzT2JzZXJ2YXRpb25Qcm92aWRlcjoKICAgICIiIkxhenksIG9uZS1pbmZlcmVuY2UtcGVyLWZyYW1lIEZPQ1VTM0Qgb2JzZXJ2YXRpb24gY2FjaGUuIiIiCgogICAgZGVmIF9faW5pdF9fKAogICAgICAgIHNlbGYsCiAgICAgICAgY2FjaGVfZGlyOiBzdHIgfCBQYXRoLAogICAgICAgICosCiAgICAgICAgc2NhbGVfdW06IHR1cGxlW2Zsb2F0LCBmbG9hdCwgZmxvYXRdID0gKDEuNjI1LCAwLjQwNjI1LCAwLjQwNjI1KSwKICAgICAgICBtYXhfZnJhbWVzOiBpbnQgPSA4MCwKICAgICkgLT4gTm9uZToKICAgICAgICBzZWxmLmNhY2hlX2RpciA9IFBhdGgoY2FjaGVfZGlyKQogICAgICAgIHNlbGYuc2NhbGVfdW0gPSB0dXBsZShmbG9hdCh2KSBmb3IgdiBpbiBzY2FsZV91bSkKICAgICAgICBzZWxmLm1heF9mcmFtZXMgPSBpbnQobWF4X2ZyYW1lcykKICAgICAgICBzZWxmLl9vYnNlcnZhdGlvbnM6IGRpY3RbdHVwbGVbc3RyLCBpbnRdLCBGb2N1c0ZyYW1lT2JzZXJ2YXRpb25dID0ge30KICAgICAgICBzZWxmLl9tb2RlbCA9IE5vbmUKICAgICAgICBzZWxmLl9jb25maWc6IFBhdGggfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYuX3dlaWdodHM6IFBhdGggfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYuX2luZmVyX3ZvbHVtZSA9IE5vbmUKICAgICAgICBzZWxmLmVycm9yOiBzdHIgfCBOb25lID0gTm9uZQogICAgICAgIHNlbGYuY2FsbHMgPSAwCgogICAgQHByb3BlcnR5CiAgICBkZWYgYXZhaWxhYmxlKHNlbGYpIC0+IGJvb2w6CiAgICAgICAgcmV0dXJuIHNlbGYuZXJyb3IgaXMgTm9uZQoKICAgIGRlZiBfbG9hZChzZWxmKSAtPiBib29sOgogICAgICAgIGlmIHNlbGYuX21vZGVsIGlzIG5vdCBOb25lOgogICAgICAgICAgICByZXR1cm4gVHJ1ZQogICAgICAgIHJvb3RzID0gWwogICAgICAgICAgICBQYXRoKG9zLmVudmlyb24uZ2V0KCJCSU9IVUJfRk9DVVMzRF9ST09UIiwgIiIpKSwKICAgICAgICAgICAgUGF0aCgiL2thZ2dsZS9pbnB1dC9kYXRhc2V0cy9xaXdlaXlpbi9mb2N1czNkLW51Y2xlaS1ydW50aW1lIiksCiAgICAgICAgICAgIFBhdGgoIi9rYWdnbGUvaW5wdXQvZm9jdXMzZC1udWNsZWktcnVudGltZSIpLAogICAgICAgIF0KICAgICAgICByb290ID0gbmV4dCgocGF0aCBmb3IgcGF0aCBpbiByb290cyBpZiBzdHIocGF0aCkgYW5kIChwYXRoIC8gImZvY3VzM2RfcnVudGltZSIpLmV4aXN0cygpKSwgTm9uZSkKICAgICAgICBpZiByb290IGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYuZXJyb3IgPSAiRk9DVVMzRCBydW50aW1lIGRhdGFzZXQgbm90IG1vdW50ZWQiCiAgICAgICAgICAgIHJldHVybiBGYWxzZQogICAgICAgIHRyeToKICAgICAgICAgICAgcnVudGltZSA9IHJvb3QgLyAiZm9jdXMzZF9ydW50aW1lIgogICAgICAgICAgICBpZiBzdHIocnVudGltZSkgbm90IGluIHN5cy5wYXRoOgogICAgICAgICAgICAgICAgc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihydW50aW1lKSkKICAgICAgICAgICAgZnJvbSBmb2N1czNkLnNlZ21lbnRhdGlvbi5GT0NVUzNELmluZmVyZW5jZV93aW4gaW1wb3J0IGJ1aWxkX3ByZWRpY3RvciwgaW5mZXJfdm9sdW1lLCBzZXR1cF9jZmcKCiAgICAgICAgICAgIGNvbmZpZyA9IHJvb3QgLyAiY29uZmlncyIgLyAiM2RfdGVzdC55YW1sIgogICAgICAgICAgICB3ZWlnaHRzID0gcm9vdCAvICJtb2RlbHMiIC8gIm1vZGVsX2ZpbmFsX251Y2xlaS5wdGgiCiAgICAgICAgICAgIGlmIG5vdCB3ZWlnaHRzLmV4aXN0cygpOgogICAgICAgICAgICAgICAgd2VpZ2h0cyA9IHJvb3QgLyAibW9kZWxzICIgLyAibW9kZWxfZmluYWxfbnVjbGVpLnB0aCIKICAgICAgICAgICAgaWYgbm90IGNvbmZpZy5leGlzdHMoKSBvciBub3Qgd2VpZ2h0cy5leGlzdHMoKToKICAgICAgICAgICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYibWlzc2luZyBjb25maWcvd2VpZ2h0cyB1bmRlciB7cm9vdH0iKQogICAgICAgICAgICBzZWxmLl9jb25maWcsIHNlbGYuX3dlaWdodHMgPSBjb25maWcsIHdlaWdodHMKICAgICAgICAgICAgc2VsZi5faW5mZXJfdm9sdW1lID0gaW5mZXJfdm9sdW1lCiAgICAgICAgICAgIHNlbGYuX21vZGVsID0gYnVpbGRfcHJlZGljdG9yKHNldHVwX2NmZyhzdHIoY29uZmlnKSwgc3RyKHdlaWdodHMpLCBkZXZpY2U9ImN1ZGEiKSkKICAgICAgICAgICAgc2VsZi5fbW9kZWwuZXZhbCgpCiAgICAgICAgICAgIHJldHVybiBUcnVlCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6ICAjIHByYWdtYTogbm8gY292ZXIgLSBLYWdnbGUtb25seSBkZXBlbmRlbmN5CiAgICAgICAgICAgIHNlbGYuZXJyb3IgPSBmIkZPQ1VTM0QgbG9hZCBmYWlsZWQ6IHt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIKICAgICAgICAgICAgcmV0dXJuIEZhbHNlCgogICAgZGVmIGdldCgKICAgICAgICBzZWxmLAogICAgICAgIGRhdGFzZXQ6IHN0ciwKICAgICAgICB0OiBpbnQsCiAgICAgICAgZnJhbWVfbG9hZGVyOiBDYWxsYWJsZVtbaW50XSwgbnAubmRhcnJheV0sCiAgICApIC0+IEZvY3VzRnJhbWVPYnNlcnZhdGlvbiB8IE5vbmU6CiAgICAgICAga2V5ID0gKHN0cihkYXRhc2V0KSwgaW50KHQpKQogICAgICAgIGlmIGtleSBpbiBzZWxmLl9vYnNlcnZhdGlvbnM6CiAgICAgICAgICAgIHJldHVybiBzZWxmLl9vYnNlcnZhdGlvbnNba2V5XQogICAgICAgIGlmIGxlbihzZWxmLl9vYnNlcnZhdGlvbnMpID49IHNlbGYubWF4X2ZyYW1lcyBvciBub3Qgc2VsZi5fbG9hZCgpOgogICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgIHRyeToKICAgICAgICAgICAgaW1wb3J0IHRpZmZmaWxlCgogICAgICAgICAgICBzZWxmLmNhY2hlX2Rpci5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICAgICAgICAgIHBhdGggPSBzZWxmLmNhY2hlX2RpciAvIGYie2RhdGFzZXR9X190e2ludCh0KTowNGR9LnRpZiIKICAgICAgICAgICAgZnJhbWUgPSBucC5hc2FycmF5KGZyYW1lX2xvYWRlcihpbnQodCkpKQogICAgICAgICAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAgICAgICAgICAgIHRpZmZmaWxlLmltd3JpdGUocGF0aCwgZnJhbWUpCiAgICAgICAgICAgIHJlc3VsdCA9IHNlbGYuX2luZmVyX3ZvbHVtZSgKICAgICAgICAgICAgICAgIGltYWdlX3BhdGg9c3RyKHBhdGgpLAogICAgICAgICAgICAgICAgY29uZmlnX2ZpbGU9c3RyKHNlbGYuX2NvbmZpZyksCiAgICAgICAgICAgICAgICB3ZWlnaHRzX3BhdGg9c3RyKHNlbGYuX3dlaWdodHMpLAogICAgICAgICAgICAgICAgbW9kZWw9c2VsZi5fbW9kZWwsCiAgICAgICAgICAgICAgICBkZXZpY2U9ImN1ZGEiLAogICAgICAgICAgICAgICAgb3V0cHV0X2Rpcj1zdHIoc2VsZi5jYWNoZV9kaXIgLyAib3V0cHV0cyIpLAogICAgICAgICAgICAgICAgel9yYXRpbz1zZWxmLnNjYWxlX3VtWzBdIC8gc2VsZi5zY2FsZV91bVsxXSwKICAgICAgICAgICAgICAgIGxvd2VyX3BlcmNlbnRpbGU9MS4wLAogICAgICAgICAgICAgICAgdXBwZXJfcGVyY2VudGlsZT05OS4wLAogICAgICAgICAgICAgICAgZGF0YV9sb2FkZXJfbnVtX3dvcmtlcnM9MCwKICAgICAgICAgICAgICAgIGNlbGxfcmFkaXVzPTE1LjAsCiAgICAgICAgICAgICAgICBiYWNrZ3JvdW5kX3RocmVzaG9sZD1mbG9hdChucC5tZWRpYW4oZnJhbWUpKSwKICAgICAgICAgICAgICAgIHN0cmlkZT1bMzIsIDk2LCA5Nl0sCiAgICAgICAgICAgICAgICBiYXRjaF9zaXplPTEyLAogICAgICAgICAgICAgICAgc2NvcmVfdGhyZXNoPTAuNiwKICAgICAgICAgICAgICAgIG1hc2tfdGhyZXNoPTAuNSwKICAgICAgICAgICAgICAgIG1pbl9lZGdlX2FyZWE9NjQsCiAgICAgICAgICAgICAgICB0b3BrX3Bvc3Rwcm9jZXNzPTMwMCwKICAgICAgICAgICAgICAgIHNhdmVfaW50ZXJtZWRpYXRlPUZhbHNlLAogICAgICAgICAgICApCiAgICAgICAgICAgIGluc3RhbmNlX21hcCA9IG5wLmFzYXJyYXkocmVzdWx0WyJpbnN0YW5jZV9tYXAiXSwgZHR5cGU9bnAuaW50MzIpCiAgICAgICAgICAgIGlmIGluc3RhbmNlX21hcC5uZGltICE9IDM6CiAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiaW5zdGFuY2VfbWFwIHNoYXBlPXtpbnN0YW5jZV9tYXAuc2hhcGV9IikKICAgICAgICAgICAgY29uZmlkZW5jZSA9IHJlc3VsdC5nZXQoImNvbmZpZGVuY2VfbWFwIikKICAgICAgICAgICAgY29uZmlkZW5jZV9tYXAgPSBOb25lIGlmIGNvbmZpZGVuY2UgaXMgTm9uZSBlbHNlIG5wLmFzYXJyYXkoY29uZmlkZW5jZSwgZHR5cGU9bnAuZmxvYXQzMikKICAgICAgICAgICAgaWYgY29uZmlkZW5jZV9tYXAgaXMgbm90IE5vbmUgYW5kIGNvbmZpZGVuY2VfbWFwLnNoYXBlICE9IGluc3RhbmNlX21hcC5zaGFwZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICAgICAgZiJjb25maWRlbmNlX21hcCBzaGFwZT17Y29uZmlkZW5jZV9tYXAuc2hhcGV9ICE9IGluc3RhbmNlX21hcCBzaGFwZT17aW5zdGFuY2VfbWFwLnNoYXBlfSIKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgbG9nX2luZm8gPSBkaWN0KHJlc3VsdC5nZXQoImxvZ19pbmZvIikgb3Ige30pCiAgICAgICAgICAgIG9ic2VydmF0aW9uID0gRm9jdXNGcmFtZU9ic2VydmF0aW9uKAogICAgICAgICAgICAgICAgZGF0YXNldD1zdHIoZGF0YXNldCksCiAgICAgICAgICAgICAgICB0PWludCh0KSwKICAgICAgICAgICAgICAgIGluc3RhbmNlX21hcD1pbnN0YW5jZV9tYXAsCiAgICAgICAgICAgICAgICBjb25maWRlbmNlX21hcD1jb25maWRlbmNlX21hcCwKICAgICAgICAgICAgICAgIGxvZ19pbmZvPWxvZ19pbmZvLAogICAgICAgICAgICApCiAgICAgICAgICAgIHNlbGYuX29ic2VydmF0aW9uc1trZXldID0gb2JzZXJ2YXRpb24KICAgICAgICAgICAgc2VsZi5jYWxscyArPSAxCiAgICAgICAgICAgIHJldHVybiBvYnNlcnZhdGlvbgogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBwcmFnbWE6IG5vIGNvdmVyIC0gS2FnZ2xlLW9ubHkgZGVwZW5kZW5jeQogICAgICAgICAgICBzZWxmLmVycm9yID0gZiJGT0NVUzNEIGluZmVyZW5jZSBmYWlsZWQgYXQge2RhdGFzZXR9L3R7dH06IHt0eXBlKGV4YykuX19uYW1lX199OiB7ZXhjfSIKICAgICAgICAgICAgcmV0dXJuIE5vbmUKCg=='))
sys.path.insert(0, '/kaggle/working')
from focus_first.observations import CenterObservation, extract_focus_instances
from focus_first.pipeline import build_frame_candidate_layers
from focus_first.audit import summarize_frame

In [ ]:
# Frozen Pilkwang detector: center observations only.
from predict_unet_transformer import PredictConfig, load_model, predict_video
from focus_first.observations import CenterObservation

device = torch.device('cuda')
weight = SUPPORT_ROOT / 'weights' / 'unet_transformer' / 'split_0' / 'edge_predictor_best.pth'
assert weight.exists(), weight
model, window, downsample = load_model(weight, device)
cfg = PredictConfig(det_threshold=0.96, det_tta=True, pool_kernel_um=3.0, use_ilp=False)
start = time.perf_counter()
coords, _ = predict_video(
    model, SAMPLE_PATH, device, cfg=cfg, window_size=window,
    max_frames=PREDICT_MAX_FRAMES, unet_batch_size=4, downsample=downsample,
)
pil_seconds = time.perf_counter() - start
coords = np.asarray(coords, dtype=np.float32)
coords = coords[np.isin(coords[:, 0].astype(np.int64), EVAL_FRAMES)]
centers_by_t = defaultdict(list)
for idx, row in enumerate(coords):
    centers_by_t[int(row[0])].append(CenterObservation(
        node_id=int(idx), t=int(row[0]), z=float(row[1]), y=float(row[2]), x=float(row[3]),
        score=0.0, source='pilkwang',
    ))
print('Pilkwang centers:', len(coords), 'seconds:', round(pil_seconds, 2))


In [ ]:
# Original FOCUS3D instance model: retain masks and confidence maps.
import tifffile

FOCUS_CONFIG = FOCUS_ROOT / 'configs' / '3d_test.yaml'
FOCUS_WEIGHTS = first_existing([
    FOCUS_ROOT / 'models' / 'model_final_nuclei.pth',
    FOCUS_ROOT / 'models ' / 'model_final_nuclei.pth',
])
runtime = str(FOCUS_ROOT / 'focus3d_runtime')
if runtime not in sys.path:
    sys.path.insert(0, runtime)
from focus3d.segmentation.FOCUS3D.inference_win import build_predictor, infer_volume, setup_cfg

focus_model = build_predictor(setup_cfg(str(FOCUS_CONFIG), str(FOCUS_WEIGHTS), device='cuda'))
focus_model.eval()
focus_instances_by_t = {}
focus_maps_by_t = {}
start = time.perf_counter()
work = Path(tempfile.mkdtemp(prefix='exp021_focus_', dir='/kaggle/working'))
try:
    for t in EVAL_FRAMES:
        t = int(t)
        frame = np.ascontiguousarray(zarr_arr[t])
        image_path = work / f'{t:04d}.tif'
        output_dir = work / f'out_{t:04d}'
        tifffile.imwrite(str(image_path), frame, metadata={'axes': 'ZYX'})
        with open(os.devnull, 'w') as sink:
            with contextlib.redirect_stdout(sink), contextlib.redirect_stderr(sink):
                result = infer_volume(
                    image_path=image_path, config_file=FOCUS_CONFIG, weights_path=FOCUS_WEIGHTS,
                    model=focus_model, device='cuda', output_dir=output_dir,
                    z_ratio=float(SCALE_UM[0] / SCALE_UM[1]), lower_percentile=1.0,
                    upper_percentile=99.0, data_loader_num_workers=0, cell_radius=15.0,
                    background_threshold=float(np.median(frame)), stride=[32, 96, 96],
                    batch_size=12, score_thresh=0.6, mask_thresh=0.5,
                    min_edge_area=64, topk_postprocess=300, save_intermediate=False,
                )
        instance_map = np.asarray(result['instance_map'], dtype=np.int32)
        confidence_map = result.get('confidence_map')
        confidence_map = None if confidence_map is None else np.asarray(confidence_map, dtype=np.float32)
        focus_maps_by_t[t] = instance_map
        focus_instances_by_t[t] = extract_focus_instances(
            instance_map, t=t, confidence_map=confidence_map,
        )
        image_path.unlink(missing_ok=True)
        shutil.rmtree(output_dir, ignore_errors=True)
finally:
    shutil.rmtree(work, ignore_errors=True)
focus_seconds = time.perf_counter() - start
print('FOCUS3D instances:', sum(map(len, focus_instances_by_t.values())), 'seconds:', round(focus_seconds, 2))


In [ ]:
# Unified nodes and source-independent continuation/division candidates.
layers = build_frame_candidate_layers(
    dict(centers_by_t), focus_instances_by_t,
    scale_um=tuple(float(v) for v in SCALE_UM), match_radius_um=MATCH_RADIUS_UM,
    max_distance_um=14.0, max_daughter_distance_um=14.0,
)
layer_rows = []
for layer in layers:
    row = summarize_frame(layer.nodes, layer.continuations_to_next, layer.divisions_to_next)
    row['t'] = int(layer.t)
    layer_rows.append(row)

def read_gt(path):
    group = zarr.open_group(str(path), mode='r')
    ids = np.asarray(group['nodes/ids'][:], dtype=np.int64)
    t = np.asarray(group['nodes/props/t/values'][:], dtype=np.int64)
    z = np.asarray(group['nodes/props/z/values'][:], dtype=np.float32)
    y = np.asarray(group['nodes/props/y/values'][:], dtype=np.float32)
    x = np.asarray(group['nodes/props/x/values'][:], dtype=np.float32)
    keep = np.isin(t, EVAL_FRAMES)
    nodes = {
        int(i): (int(tt), float(zz), float(yy), float(xx))
        for i, tt, zz, yy, xx, selected in zip(ids, t, z, y, x, keep)
        if selected
    }
    edges = [
        tuple(map(int, row)) for row in np.asarray(group['edges/ids'][:], dtype=np.int64)
        if int(row[0]) in nodes and int(row[1]) in nodes
    ]
    return nodes, edges

gt_nodes, gt_edges = read_gt(GT_PATH)
proposal_nodes = [node for layer in layers for node in layer.nodes]
by_t = defaultdict(list)
for node in proposal_nodes:
    by_t[int(node.t)].append(node)

def nearest(value):
    tt, zz, yy, xx = value
    choices = by_t.get(int(tt), [])
    if not choices:
        return None
    target = np.asarray((zz, yy, xx), dtype=np.float64) * SCALE_UM
    distances = [np.linalg.norm(np.asarray(node.point) * SCALE_UM - target) for node in choices]
    index = int(np.argmin(distances))
    return choices[index] if float(distances[index]) <= MATCH_RADIUS_UM else None

mapped = {node_id: nearest(value) for node_id, value in gt_nodes.items()}
outgoing = defaultdict(list)
for source_id, target_id in gt_edges:
    outgoing[source_id].append(target_id)
division_rows = []
for parent_id, children in sorted(outgoing.items()):
    if len(children) < 2:
        continue
    parent, d1, d2 = (mapped.get(parent_id), mapped.get(children[0]), mapped.get(children[1]))
    found = False
    if parent is not None and d1 is not None and d2 is not None:
        for layer in layers:
            triples = {(c.parent_id, c.daughter1_id, c.daughter2_id) for c in layer.divisions_to_next}
            found |= (parent.proposal_id, d1.proposal_id, d2.proposal_id) in triples
            found |= (parent.proposal_id, d2.proposal_id, d1.proposal_id) in triples
    division_rows.append({
        'gt_parent': int(parent_id), 'gt_daughters': [int(v) for v in children[:2]],
        'parent_detected': int(parent is not None), 'daughter1_detected': int(d1 is not None),
        'daughter2_detected': int(d2 is not None), 'complete_candidate': int(found),
    })

report = {
    'experiment': 'EXP021', 'stage': 'candidate_layer_only', 'sample_id': SAMPLE_ID,
    'eval_frames': [int(v) for v in EVAL_FRAMES],
    'runtime_seconds': {'pilkwang': pil_seconds, 'focus3d': focus_seconds},
    'node_counts': {
        'gt_nodes': len(gt_nodes), 'proposal_nodes': len(proposal_nodes),
        'consensus': sum(node.kind == 'consensus' for node in proposal_nodes),
        'pilkwang_only': sum(node.kind == 'pilkwang_only' for node in proposal_nodes),
        'focus_only': sum(node.kind == 'focus_only' for node in proposal_nodes),
        'matched_gt_nodes': sum(value is not None for value in mapped.values()),
    },
    'division_rows': division_rows, 'formal_cv': None, 'submission_generated': False,
}
out = Path('/kaggle/working/exp021_focus_first_v1')
out.mkdir(parents=True, exist_ok=True)
(out / 'REPORT.json').write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
(out / 'LAYER_SUMMARY.json').write_text(json.dumps(layer_rows, indent=2, default=str), encoding='utf-8')
(out / 'DIVISION_CANDIDATES.json').write_text(json.dumps(division_rows, indent=2, default=str), encoding='utf-8')
print(json.dumps(report, indent=2, default=str))
